# 第 26 课：量化基础——INT8、PTQ、QAT 与误差

量化不是简单把 `float32` 强制转成整数，而是用 scale 和 zero-point 建立浮点值与有限整数网格的映射。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 量化与部署 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 25 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | scale/zero-point、PTQ/QAT、per-tensor/per-channel |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：scale/zero-point、PTQ/QAT、per-tensor/per-channel。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [ ]:
from pathlib import Path
import time
import sys
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT=find_root(); ARTIFACTS=ROOT/"artifacts";ARTIFACTS.mkdir(exist_ok=True)
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

from ipywidgets import interact, IntSlider, FloatSlider

## 1. Affine quantization

$$q=\operatorname{clamp}(\operatorname{round}(x/s)+z,q_{min},q_{max})$$
$$\hat{x}=s(q-z)$$

`q` 是整数，$\hat{x}$ 是反量化后的近似值。量化误差来自舍入和截断。

In [ ]:
def symmetric_quantize(x,bits=8):
    qmax=2**(bits-1)-1;scale=max(np.max(np.abs(x))/qmax,1e-12)
    q=np.clip(np.round(x/scale),-qmax,qmax).astype(np.int32)
    return q,q.astype(np.float32)*scale,scale

x=np.linspace(-2,2,401,dtype=np.float32)
for bits in [8,4,2]:
    q,xhat,s=symmetric_quantize(x,bits)
    print(bits,"bits scale",s,"max error",np.max(np.abs(x-xhat)),"levels",len(np.unique(q)))

## 2. 交互观察 bit 数与离群值

In [ ]:
@interact(bits=IntSlider(min=2,max=8,value=4),outlier=FloatSlider(min=1,max=20,value=2,step=1))
def show(bits=4,outlier=2):
    values=np.concatenate([np.linspace(-1,1,200),[outlier]]).astype(np.float32)
    q,xhat,scale=symmetric_quantize(values,bits)
    plt.scatter(values,xhat,s=12);plt.plot([values.min(),values.max()],[values.min(),values.max()],color="C1")
    plt.xlabel("FP32 value");plt.ylabel("Dequantized value");plt.title(f"{bits}-bit quantization, scale={scale:.4f}");plt.show()
    print("mean abs error",np.mean(np.abs(values-xhat)))

离群值会扩大 scale，使大量普通值挤在更粗的网格上。校准集、percentile clipping 和 per-channel quantization因此非常重要。

## 3. Per-tensor 与 per-channel

In [ ]:
rng=np.random.default_rng(2)
W=np.vstack([rng.normal(0,.05,100),rng.normal(0,.5,100),rng.normal(0,3,100)]).astype(np.float32)
_,Wt,_=symmetric_quantize(W,8)
Wc=np.empty_like(W)
for i,row in enumerate(W): _,Wc[i],_=symmetric_quantize(row,8)
print("per-tensor MAE",np.mean(np.abs(W-Wt)))
print("per-channel MAE",np.mean(np.abs(W-Wc)))

## 4. PTQ、动态量化、静态量化、QAT

- Dynamic PTQ：权重预先量化，activation 参数运行时计算；容易上手，常用于 RNN/Transformer。
- Static PTQ：用代表性 calibration data 提前统计 activation 范围；运行开销更低，但更依赖校准质量。
- QAT：训练时模拟量化误差，让模型适应；成本更高，通常在 PTQ 精度不达标时使用。
- Weight-only：只量化权重，节省模型大小/带宽，但 activation 仍较高精度。

当前 PyTorch 量化能力正在集中到 `torchao`；后续代码避免把即将迁移的旧 eager API 当成唯一方案。

## 5. ASR 量化必须检查什么

- Logit/CTC posterior 差异；
- Greedy/beam 输出变化；
- CER/WER，而不只是 MSE；
- 长音频 cache 漂移；
- 热词和语言模型融合后的排序变化；
- 模型大小、峰值内存、RTF、P99；
- 目标 CPU/GPU/移动硬件是否有对应低精度 kernel。

## 本课测试

1. scale 越小是否永远越好？
2. 离群值怎样影响 per-tensor 量化？
3. static PTQ 为什么需要 calibration data？
4. QAT 是否应当永远作为第一选择？
5. 模型缩小 4 倍是否保证推理快 4 倍？

<details><summary>展开参考答案</summary>

1. 不是，过小会截断大值。2. 扩大范围和 scale，使普通值误差变粗。3. 提前估计 activation 范围。4. 不，应先尝试成本低的 PTQ 并验证。5. 不保证，取决于低精度 kernel、内存、算子覆盖和量化/反量化开销。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 26 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `scale/zero-point`、`PTQ/QAT`、`per-tensor/per-channel`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**离群值扩大 activation 范围**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**实现对称量化并比较 bit 数和量化误差**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**连接量化误差与 CTC 排名变化**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：scale/zero-point、PTQ/QAT、per-tensor/per-channel。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 scale/zero-point、PTQ/QAT、per-tensor/per-channel。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
